# 📊 Exploratory Data Analysis (EDA)
## Hull Tactical Market Prediction - Phase 1

**Date:** 2025-11-11

**Objectives:**
1. Feature Group Analysis (M/E/I/P/V/S/D)
2. Target Variable Time Series Characteristics  
3. Correlation Analysis
4. Market Regime Identification
5. Train/Test Distribution Shift Detection
6. Feature Stationarity Assessment
7. Benchmark Calculation

## 1. Setup and Imports

In [ ]:
# Standard libraries
import sys
import os
import warnings
warnings.filterwarnings('ignore')

# Add project root to path
sys.path.append(os.path.dirname(os.path.dirname(os.path.abspath('__file__'))))

# Data manipulation
import numpy as np
import pandas as pd

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Statistical tests
from scipy import stats
from scipy.stats import ks_2samp
from statsmodels.tsa.stattools import adfuller, acf, pacf
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

# Clustering
from sklearn.cluster import KMeans

# Project utilities
from src.data import DataLoader, calculate_benchmark_score
from src.utils import set_seed

# Set random seed
set_seed(42)

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.4f}'.format)

# Plot settings  
plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.figsize'] = (14, 6)

print('✅ Setup complete!')

## 2. Load Data

In [ ]:
# Initialize DataLoader
loader = DataLoader()

# Load data
train_df, test_df = loader.load_data()

print(f'📦 Train: {train_df.shape}, Test: {test_df.shape}')
print(f'📅 Date range: {train_df["date_id"].min()} - {train_df["date_id"].max()}')

# Display feature groups
print('\n🏷️ Feature Groups:')
for group, features in loader.feature_groups.items():
    print(f'  {group}: {len(features)} features')

train_df.head()

## 3. Data Quality Check

In [ ]:
# Quality check
quality = loader.check_data_quality(train_df)

print(f"Total rows: {quality['total_rows']}")
print(f"Duplicates: {quality['duplicates']}")
print(f"Date gaps: {len(quality['date_gaps'])}")

print("\nMissing values by group:")
for group, stats in quality['missing_summary'].items():
    print(f"  {group}: {stats['avg_missing_pct']:.1f}% avg missing")

## 4. Feature Group Analysis

In [ ]:
# Analyze each feature group
for group, features in loader.feature_groups.items():
    if not features or group == 'D':
        continue
    
    print(f"\n{'='*60}\n{group} Features ({len(features)})\n{'='*60}")
    
    # Descriptive statistics
    desc = train_df[features].describe()
    print(desc)
    
    # Plot sample distributions
    sample_features = features[:min(4, len(features))]
    fig, axes = plt.subplots(1, len(sample_features), figsize=(16, 4))
    if len(sample_features) == 1:
        axes = [axes]
    
    for idx, feat in enumerate(sample_features):
        train_df[feat].hist(bins=50, ax=axes[idx], alpha=0.7)
        axes[idx].set_title(f'{feat} Distribution')
    
    plt.suptitle(f'{group} Group - Sample Distributions')
    plt.tight_layout()
    plt.show()

## 5. Target Variable Analysis

In [ ]:
target = 'forward_returns'

# Time series plot
plt.figure(figsize=(16, 6))
plt.plot(train_df['date_id'], train_df[target], alpha=0.6, linewidth=0.5)
plt.title('Forward Returns Over Time')
plt.xlabel('Date ID')
plt.ylabel('Forward Returns')
plt.grid(True, alpha=0.3)
plt.show()

# Distribution plots
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Histogram
axes[0].hist(train_df[target].dropna(), bins=100, alpha=0.7, edgecolor='black')
axes[0].set_title('Distribution')
axes[0].set_xlabel('Forward Returns')

# Q-Q plot
stats.probplot(train_df[target].dropna(), dist="norm", plot=axes[1])
axes[1].set_title('Q-Q Plot')

# Box plot
axes[2].boxplot(train_df[target].dropna())
axes[2].set_title('Box Plot')
axes[2].set_ylabel('Forward Returns')

plt.tight_layout()
plt.show()

# Statistics
print(f"Mean: {train_df[target].mean():.6f}")
print(f"Std: {train_df[target].std():.6f}")
print(f"Skewness: {stats.skew(train_df[target].dropna()):.4f}")
print(f"Kurtosis: {stats.kurtosis(train_df[target].dropna()):.4f}")

## 6. Autocorrelation Analysis

In [ ]:
# ACF and PACF plots
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

plot_acf(train_df[target].dropna(), lags=50, ax=axes[0])
axes[0].set_title('Autocorrelation Function (ACF)')

plot_pacf(train_df[target].dropna(), lags=50, ax=axes[1])
axes[1].set_title('Partial Autocorrelation Function (PACF)')

plt.tight_layout()
plt.show()

# Print significant lags
acf_vals = acf(train_df[target].dropna(), nlags=20)
print("Significant ACF lags (|r| > 0.05):")
for i, val in enumerate(acf_vals[1:], 1):
    if abs(val) > 0.05:
        print(f"  Lag {i}: {val:.4f}")

## 7. Correlation Analysis

In [ ]:
# Sample features for correlation
sample_features = []
for group, features in loader.feature_groups.items():
    if group != 'D' and features:
        sample_features.extend(features[:min(5, len(features))])

sample_features.append(target)

# Correlation matrix
corr_matrix = train_df[sample_features].corr()

# Plot heatmap
plt.figure(figsize=(14, 12))
sns.heatmap(corr_matrix, cmap='coolwarm', center=0, 
            square=True, linewidths=0.5, vmin=-1, vmax=1)
plt.title('Feature Correlation Heatmap (Sample)')
plt.tight_layout()
plt.show()

# Top correlations with target
target_corr = corr_matrix[target].drop(target).sort_values(ascending=False)
print("\nTop 10 Positive Correlations with Target:")
print(target_corr.head(10))
print("\nTop 10 Negative Correlations with Target:")
print(target_corr.tail(10))

## 8. Market Regime Analysis

In [ ]:
# Calculate rolling volatility
window = 20
train_df['rolling_vol'] = train_df[target].rolling(window).std()

# Identify regimes (quartiles)
vol_quartiles = train_df['rolling_vol'].quantile([0.25, 0.5, 0.75])
print(f"Volatility Quartiles:\n{vol_quartiles}")

# Classify regimes
def classify_regime(vol):
    if pd.isna(vol):
        return 'Unknown'
    elif vol < vol_quartiles[0.25]:
        return 'Low Vol'
    elif vol < vol_quartiles[0.75]:
        return 'Medium Vol'
    else:
        return 'High Vol'

train_df['regime'] = train_df['rolling_vol'].apply(classify_regime)

# Plot regimes
fig, axes = plt.subplots(2, 1, figsize=(16, 10), sharex=True)

# Returns with regime coloring
for regime in ['Low Vol', 'Medium Vol', 'High Vol']:
    mask = train_df['regime'] == regime
    axes[0].scatter(train_df.loc[mask, 'date_id'], 
                   train_df.loc[mask, target],
                   label=regime, alpha=0.5, s=1)
axes[0].set_ylabel('Forward Returns')
axes[0].legend()
axes[0].set_title('Returns by Volatility Regime')
axes[0].grid(True, alpha=0.3)

# Rolling volatility
axes[1].plot(train_df['date_id'], train_df['rolling_vol'], linewidth=1)
axes[1].axhline(vol_quartiles[0.25], color='g', linestyle='--', label='Q1')
axes[1].axhline(vol_quartiles[0.75], color='r', linestyle='--', label='Q3')
axes[1].set_xlabel('Date ID')
axes[1].set_ylabel('Rolling Volatility')
axes[1].legend()
axes[1].set_title(f'Rolling Volatility ({window}-day window)')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Performance by regime
print("\nPerformance by Regime:")
regime_stats = train_df.groupby('regime')[target].agg(['count', 'mean', 'std', 'min', 'max'])
print(regime_stats)

## 9. Stationarity Tests

In [ ]:
# Test stationarity for target and sample features
test_features = [target] + [f for g, feats in loader.feature_groups.items() 
                            for f in feats[:2] if g != 'D' and feats][:10]

stationarity_results = []

for feat in test_features:
    series = train_df[feat].dropna()
    
    if len(series) < 50:
        continue
    
    # ADF test
    adf_result = adfuller(series, autolag='AIC')
    
    stationarity_results.append({
        'Feature': feat,
        'ADF Statistic': adf_result[0],
        'ADF p-value': adf_result[1],
        'Stationary (p<0.05)': adf_result[1] < 0.05
    })

stationarity_df = pd.DataFrame(stationarity_results)
print("Stationarity Test Results (ADF):")
print(stationarity_df.sort_values('ADF p-value'))

# Count stationary vs non-stationary
n_stationary = stationarity_df['Stationary (p<0.05)'].sum()
n_total = len(stationarity_df)
print(f"\nStationary: {n_stationary}/{n_total} ({n_stationary/n_total*100:.1f}%)") 

## 10. Train/Test Distribution Shift

In [ ]:
# Compare distributions for common features
common_features = [c for c in train_df.columns if c in test_df.columns 
                   and c not in ['date_id', 'is_scored']]

# Sample features to test
test_sample = [f for f in common_features if not f.startswith('lagged')][:15]

shift_results = []

for feat in test_sample:
    train_vals = train_df[feat].dropna()
    test_vals = test_df[feat].dropna()
    
    if len(train_vals) < 10 or len(test_vals) < 5:
        continue
    
    # KS test
    ks_stat, ks_pval = ks_2samp(train_vals, test_vals)
    
    shift_results.append({
        'Feature': feat,
        'KS Statistic': ks_stat,
        'KS p-value': ks_pval,
        'Significant Shift (p<0.05)': ks_pval < 0.05
    })

shift_df = pd.DataFrame(shift_results)
print("Distribution Shift Analysis (KS Test):")
print(shift_df.sort_values('KS Statistic', ascending=False))

# Count shifts
n_shifts = shift_df['Significant Shift (p<0.05)'].sum()
print(f"\nFeatures with significant shift: {n_shifts}/{len(shift_df)}")

## 11. Benchmark Calculation

In [ ]:
# Calculate benchmark with allocation=1.0
score, returns, metrics = calculate_benchmark_score(train_df, allocation=1.0)

print("=" * 60)
print("BENCHMARK METRICS (Allocation = 1.0)")
print("=" * 60)
print(f"\nScore: {score:.4f}")
print(f"Sharpe Ratio: {metrics['sharpe']:.4f}")
print(f"Annual Sharpe: {metrics['annual_sharpe']:.4f}")
print(f"Volatility Penalty: {metrics['vol_penalty']:.4f}")
print(f"\nAnnual Return: {metrics['annual_return']:.2%}")
print(f"Annual Volatility: {metrics['annual_vol']:.2%}")
print(f"Max Drawdown: {metrics['max_drawdown']:.2%}")
print(f"Win Rate: {metrics['win_rate']:.2%}")

# Cumulative returns plot
cumulative_returns = (1 + returns).cumprod()

fig, axes = plt.subplots(2, 1, figsize=(16, 10))

# Cumulative returns
axes[0].plot(cumulative_returns.values, linewidth=1.5)
axes[0].set_title('Benchmark Strategy - Cumulative Returns (Allocation=1.0)')
axes[0].set_ylabel('Cumulative Return')
axes[0].grid(True, alpha=0.3)
axes[0].axhline(y=1.0, color='r', linestyle='--', alpha=0.5)

# Drawdown
running_max = cumulative_returns.expanding().max()
drawdown = (cumulative_returns - running_max) / running_max

axes[1].fill_between(range(len(drawdown)), drawdown.values, 0, alpha=0.3, color='red')
axes[1].plot(drawdown.values, linewidth=1, color='darkred')
axes[1].set_title('Drawdown')
axes[1].set_xlabel('Days')
axes[1].set_ylabel('Drawdown')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 12. Summary and Insights

In [ ]:
print("="*60)
print("EDA SUMMARY")
print("="*60)

print(f"\n📦 Data Shape:")
print(f"  Train: {train_df.shape}")
print(f"  Test: {test_df.shape}")

print(f"\n🏷️ Feature Groups:")
for group, features in loader.feature_groups.items():
    print(f"  {group}: {len(features)} features")

print(f"\n📊 Target Statistics:")
print(f"  Mean: {train_df[target].mean():.6f}")
print(f"  Std: {train_df[target].std():.6f}")
print(f"  Sharpe: {train_df[target].mean()/train_df[target].std():.4f}")

print(f"\n🎯 Benchmark Performance:")
print(f"  Score: {score:.4f}")
print(f"  Annual Sharpe: {metrics['annual_sharpe']:.4f}")
print(f"  Max Drawdown: {metrics['max_drawdown']:.2%}")

print(f"\n⚠️ Key Findings:")
print(f"  - Stationary features: {n_stationary}/{n_total}")
print(f"  - Distribution shifts: {n_shifts}/{len(shift_df)}")
print(f"  - Missing data: See quality check above")

print("\n✅ EDA Complete!")